### Imports

In [17]:
import cv2
import mediapipe as mp
from pathlib import Path
import os
import numpy as np
import time
import math


In [18]:
import tensorflow as tf
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Conv3D, LSTM, Dense, Dropout, Bidirectional, MaxPool3D, Activation, Reshape, SpatialDropout3D, BatchNormalization, TimeDistributed, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras import layers, models

### Global Variables

### Face and Mouth Detection

In [19]:
LIP_LAMDMARKS = [61, 185, 40, 39, 37, 0, 267, 269, 270, 409, 291, 78, 191, 80, 81, 82, 13, 312, 311, 310, 415, 308, 95, 88, 178, 87, 14, 317, 402, 318, 324, 146, 91, 181, 84, 17, 314, 405, 321, 375]    
# 0 = Top middle lip
TEST_IMAGE = cv2.imread('/home/tymstr/Documents/GitHub/audiovisual-transcription/TestItems/test_two_ppl_img.png', cv2.IMREAD_COLOR)
mpDraw = mp.solutions.drawing_utils
mpFaceMesh = mp.solutions.face_mesh
faceMesh = mpFaceMesh.FaceMesh(max_num_faces=3)
drawSpec = mpDraw.DrawingSpec(thickness=1, circle_radius=1)


def FindMouth(img):
    points = []
    dict = {}
    results = faceMesh.process(img)
    if results.multi_face_landmarks:
        for faceLms in results.multi_face_landmarks:
            for id,lm in enumerate(faceLms.landmark):
                if id == 0:
                    image_height, image_width, image_channels = img.shape
                    x,y = int(lm.x*image_width), int(lm.y*image_height)
                    points += [(x,y,id)]
    return points

FindMouth(TEST_IMAGE)


I0000 00:00:1744341960.781743   15989 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1744341960.871608   16518 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 550.120), renderer: NVIDIA GeForce RTX 4070/PCIe/SSE2
W0000 00:00:1744341960.876701   16516 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1744341960.888171   16510 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[(145, 289, 0), (645, 288, 0)]

##### Show test image

#### Draw on Frames

In [20]:
def draw_box_around_point(img, x, y, color=(0, 255, 0), thickness=2):
    """Draws a small box around the given (x, y) point on the image."""
    top_left = (x - 60 // 2, y - 30 // 2)
    bottom_right = (x + 40 // 2, y + 70 // 2)
    
    cv2.rectangle(img, top_left, bottom_right, color, thickness)

#### Model

In [21]:
input_shape = (10, 100, 100, 3)
model5 = tf.keras.models.Sequential([
    # Input Layer
    layers.Input(shape=input_shape),

    # (2+1)D Conv Block 1
    layers.Conv3D(32, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(32, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(1, 2, 2)),

    # (2+1)D Conv Block 2
    layers.Conv3D(64, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(64, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(2, 2, 2)),

    # (2+1)D Conv Block 3
    layers.Conv3D(128, kernel_size=(1, 3, 3), padding='same', activation='relu'),
    layers.Conv3D(128, kernel_size=(3, 1, 1), padding='same', activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling3D(pool_size=(2, 2, 2)),

    # Global Feature Aggregation
    layers.GlobalAveragePooling3D(),
    layers.Dropout(0.5),

    # Fully Connected Layer
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    # Output Layer (Binary Classification)
    layers.Dense(1, activation='sigmoid')  # Change from softmax to sigmoid
])

model5.load_weights('./finaltest.weights.h5')

print(model5.input_shape)

(None, 10, 100, 100, 3)


In [22]:
predict_frames = []


def PredictFrames():
    results = np.array(predict_frames[-10:], dtype=np.float32)[..., [2,1,0]]
    input = np.expand_dims(results, axis=0)
    prediction = model5.predict(input)
    return prediction[0][0]


def PredictFramesfromArray(array):
    results = np.array(array, dtype=np.float32)[..., [2,1,0]]
    input = np.expand_dims(results, axis=0)
    prediction = model5.predict(input)
    return prediction[0][0]

### Main Function

#### Capture frames and audio and displays frames and annotations

In [ ]:
def main(video):
    # Open the default camera (0 is usually the built-in webcam)
    cap = cv2.VideoCapture(video)
    


    if not cap.isOpened():
        print("Error: Could not open camera.")
        return
    
    frame_number = 0

    current_prediction = 0

    while True:
        # Capture frame-by-frame
        ret, frame = cap.read()
        
        if not ret:
            print("Error: Couldn't read frame.")
            break
        
        frame_number += 1

        # cv2.putText(frame, f"Frame: {frame_number}", (10, 50),
        #             cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
        ##Try draw around mouth
        try:
            mouths = FindMouth(frame)
            for x,y,z in mouths:
                # cv2.circle(frame, (x,y), 1, (0,0,255), 1)
                draw_box_around_point(frame, x, y)
                new_frame = frame[y-40:y+60,x-50:x+50,:]
                new_frame = tf.image.convert_image_dtype(new_frame, tf.float32)
                new_frame = tf.image.resize_with_pad(new_frame,100,100)
                predict_frames.append(new_frame)
                # prediction = PredictFrames(new_frame)
                # if prediction != 0:
                #     current_prediction = prediction
        except:
            cv2.putText(frame, "No mouth identified", (300, 400), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
            pass



        # cv2.putText(frame, str(current_prediction), (250, 350),cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2 )
            
        # Add text overlay
        text = "Live Camera Feed"
        if len(predict_frames) > 10:
            current_prediction = PredictFrames()
            if current_prediction > 0.5:
                text = "talking"
            else:
                text = "not talking"
        cv2.putText(frame, str(text), (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
        
        # Display the resulting frame
        cv2.imshow('Camera Feed', frame)

        if cv2.waitKey(1) & 0xFF == ord('p'):
            current_prediction = PredictFrames()
            print("prediction:" , current_prediction)

        
        # Exit when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Release the capture and close the window
    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main("bbaf2n.mpg")
    time.sleep(5)

    main("bbad5n.mpg")

    # cap = cv2.VideoCapture("bbaf2n.mpg")
    # # list = []
    # for i in range(len(predict_frames)):
    #     # list += [i, str(PredictFramesfromArray(predict_frames[-i:]))]
        
    #     ret, frame = cap.read()
    #     if not ret:
    #         break

    #     if i > 10:
    #         cv2.putText(frame, str(PredictFramesfromArray(predict_frames[-i:])) + str(i), (50, 50),cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2 )
    #         input("next frame")
    #     cv2.imshow('frame', frame)

    #     if cv2.waitKey(1) & 0xFF == ord('q'):
    #         break

    # cap.release()
    # cv2.destroyAllWindows()








    


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 694ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━

In [16]:
test_samples = []
example_input = np.random.rand(1, 10, 100, 100, 3).astype(np.float32)
# example_frames = next(iter(test_ds))

single_frame = np.random.rand(100, 100, 3).astype(np.float32)

# Repeat the frame 10 times to create a sequence
video_clip = np.tile(single_frame, (10, 1, 1, 1))  # shape will be (10, 100, 100, 3)

# Now add the batch dimension, resulting in shape (1, 10, 100, 100, 3)
example_input2 = np.expand_dims(video_clip, axis=0)

for i in range(1):
    prediction = model5.predict(example_input2)
    # next(iter(train_ds))
    print(prediction[0][0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 671ms/step
0.004026134
